In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.svm import SVR, SVC
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import cross_val_score
import warnings
warnings.filterwarnings('ignore')

In [3]:
X_train = pd.read_csv('../Datos/preparados/TrainX.csv')
y_train = pd.read_csv('../Datos/preparados/TrainY.csv')
X_val = pd.read_csv('../Datos/preparados/ValidationX.csv') 
y_val = pd.read_csv('../Datos/preparados/ValidationY.csv')
X_test = pd.read_csv('../Datos/preparados/TestX.csv')
y_test = pd.read_csv('../Datos/preparados/TestY.csv')

In [42]:
def crear_categorias_rentabilidad(y):
    return (y >= 1).astype(int)

In [43]:
y_train_class = crear_categorias_rentabilidad(y_train)
y_val_class = crear_categorias_rentabilidad(y_val)
y_test_class = crear_categorias_rentabilidad(y_test)

print("Distribucion de clases:")
print(f" Rentabilidad baja: {(y_train_class == 0).sum()}")
print(f" Rentabilidad alta: {(y_train_class == 1).sum()}")


Distribucion de clases:
 Rentabilidad baja: ROI    186508
dtype: int64
 Rentabilidad alta: ROI    274592
dtype: int64


Para nuestro problema de predicción de rentabilidad cinematográfica, seleccionamos:

1. REGRESIÓN: Random Forest Regressor
   Debido a su consistencia y capacidad de manejar relaciones no lineales, ademas de ser ideal para problemas de regresion en datasets complejos donde no son evidentes las relaciones entre variables. Su naturaleza de conjunto lo vuelve resistente al sobreajuste y ofrece generalizaciones adecuadas.

2. ERROR: Gradient Boosting Regressor
   Gracias a su eficiencia computacional con datasets grandes, Gradient Boosting escala linealmente. Además, captura patrones complejos mediante boosting secuencial, ofreciendo alto rendimiento predictivo con tiempos de entrenamiento razonables.
 
3. CLASIFICACIÓN: Random Forest Classifier
   Transformar el problema a una clasificacion binaria (rentabilidad baja o alta). Ofreciendo interpretabilidad de resultados mediante probabilidades de clase, ademas de ser robusto frente a desbalanceos, complementando el analisis de regresion.

Error principal: rmse (root mean square error)
Raíz cuadrada del promedio de las diferencias al cuadrado entre los valores predichos y los valores reales.
Por que se uso esta metrica:
	•	Es facil de entender porque usa las mismas unidades que la variable que se quiere predecir (roi).
	•	Da más peso a los errores grandes, lo cual es importante en decisiones de inversion, ya que sobreestimar el roi puede causar perdidas.
	•	Funciona bien en los procesos de optimizacion del modelo.
	•	Es una medida común en muchos estudios, lo que permite comparar resultados con otros trabajos.

Otras metricas que se usaron:
	•	mae (mean absolute error): sirve para ver que tan estable es el modelo frente a valores muy distintos.
	•	r² (coeficiente de determinacion): muestra que porcentaje de la variación del roi logra explicar el modelo.
	•	accuracy: se usa en clasificación para ver que tan bien acierta el modelo al decir si una película fue rentable o no.

Variables independientes (features del dataset)
Caracteristicas tecnicas y comerciales de las peliculas:
• Presupuesto (budgetusd)
• Taquilla global (global_boxofficeusd)
• Ventas primera semana (one_week_salesusd)
• Ratings de criticas (imdbrating, rottentomatoesscore)
• Variables categoricas codificadas (genero, pais)
• Metricas derivadas (roi_ratio, categorias de rentabilidad)
Estas son medidas objetivas que se conocen antes del estreno y que pueden influir en el roi.

Factores controlables (hiperparametros)
Parametros de los modelos que se ajustaron durante las pruebas:
• random forest: n_estimators, max_depth, min_samples_split
• gradient boosting: n_estimators, max_depth, learning_rate
• random forest classifier: n_estimators, max_depth, min_samples_split
Son decisiones sobre como funciona el modelo y se pueden cambiar para mejorar su rendimiento.

Factores no controlables
Variables externas que afectan el roi pero no estan en el dataset:
• Campanas de marketing que no se midieron
• Temporada o momento del estreno
• Competencia en cartelera
• Eventos globales o situaciones economicas
• Cambios en los gustos del publico
• Opiniones informales y marketing boca a boca
Son factores que no se pueden medir del todo y que explican parte del error del modelo.

Variable dependiente
roi (return on investment)
• Formato continuo: usado en regresion (valor numerico)
• Formato categorico: usado en clasificacion (rentabilidad alta ≥ 1, baja < 1)
Es la medida de exito que se busca predecir y muestra el retorno financiero de la inversion en la pelicula.

Algoritmo 1: random forest regressor
Este modelo usa muchos arboles para hacer predicciones y promedia sus resultados.
• n_estimators: cuantos arboles se crean.
• max_depth: que tan profundo puede crecer cada arbol.
• min_samples_split: cuantos datos minimos necesita un nodo para dividirse.

Algoritmo 2: gradient boosting regressor
Tambien usa arboles, pero los va construyendo uno por uno, corrigiendo los errores del anterior.
• n_estimators: cuantos pasos o arboles se usan.
• max_depth: hasta que punto puede crecer cada arbol.
• learning_rate: que tanto aprende el modelo en cada paso (si es muy alto, puede pasarse; si es muy bajo, tarda mas).

Algoritmo 3: random forest classifier
Es parecido al primero, pero se usa para clasificar, por ejemplo, si una pelicula fue rentable o no.
• n_estimators: numero de arboles que se crean.
• max_depth: profundidad maxima de los arboles.
• min_samples_split: minimo de datos que necesita un nodo para dividirse.


Random forest (regressor y classifier)
Se probaron distintas combinaciones para encontrar un buen equilibrio entre precision y tiempo de ejecucion:
• n_estimators: [50, 100] : cantidad de arboles del modelo.
• max_depth: [5, 10] : controla que tan profundo crece cada arbol (mas profundidad puede causar sobreajuste).
• min_samples_split: [2, 5] : numero minimo de datos que necesita un nodo para dividirse.

Gradient boosting regressor
Tambien se probaron diferentes configuraciones para ajustar su comportamiento:
• n_estimators: [50, 100] : cantidad de pasos o arboles que usa el modelo.
• max_depth: [3, 6] : profundidad moderada para evitar que aprenda de mas.
• learning_rate: [0.1, 0.05] : velocidad con la que el modelo aprende en cada paso.

Estrategia de busqueda
Cada algoritmo se probo con 8 combinaciones distintas de parametros, lo que dio un total de 24 configuraciones entre los tres modelos.
Por que se hizo asi:
Esta forma de probar permitio explorar distintas opciones sin que el proceso tardara demasiado, y ayudo a ver como cambian los resultados cuando se ajustan los parametros principales.


In [31]:
n_estimators_list = [50, 100]      # 2 valores en lugar de 3
max_depth_list = [5, 10]           # 2 valores en lugar de 3  
min_samples_split_list = [2, 5]    # 2 valores en lugar de 3

print("Hiperparametros optimizados (2×2×2 = 8 combinaciones):")
print(f"   n_estimators: {n_estimators_list}")
print(f"   max_depth: {max_depth_list}")
print(f"   min_samples_split: {min_samples_split_list}")

results_rf = []

for n_estimators in n_estimators_list:
    for max_depth in max_depth_list:
        for min_samples_split in min_samples_split_list:
            print(f"Probando: n_estimators={n_estimators}, max_depth={max_depth}, min_samples_split={min_samples_split}")
            
            try:
                model = RandomForestRegressor(
                    n_estimators=n_estimators,
                    max_depth=max_depth,
                    min_samples_split=min_samples_split,
                    random_state=42,
                    n_jobs=-1,
                    verbose=0
                )
                model.fit(X_train, y_train.values.ravel())
                
                y_train_pred = model.predict(X_train)
                y_val_pred = model.predict(X_val)
                
                train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
                val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
                train_r2 = r2_score(y_train, y_train_pred)
                val_r2 = r2_score(y_val, y_val_pred)
                
                results_rf.append({
                    'algorithm': 'RandomForest_Regressor',
                    'n_estimators': n_estimators,
                    'max_depth': max_depth,
                    'min_samples_split': min_samples_split,
                    'train_rmse': train_rmse,
                    'val_rmse': val_rmse,
                    'train_r2': train_r2,
                    'val_r2': val_r2
                })
                
                print(f"   Train RMSE: {train_rmse:.4f}, Val RMSE: {val_rmse:.4f}")
                
            except Exception as e:
                print(f"   Error: {e}")
                continue

Hiperparametros optimizados (2×2×2 = 8 combinaciones):
   n_estimators: [50, 100]
   max_depth: [5, 10]
   min_samples_split: [2, 5]
Probando: n_estimators=50, max_depth=5, min_samples_split=2
   Train RMSE: 16.6295, Val RMSE: 16.3805
Probando: n_estimators=50, max_depth=5, min_samples_split=5
   Train RMSE: 16.6295, Val RMSE: 16.3805
Probando: n_estimators=50, max_depth=10, min_samples_split=2
   Train RMSE: 13.2612, Val RMSE: 15.2895
Probando: n_estimators=50, max_depth=10, min_samples_split=5
   Train RMSE: 13.4423, Val RMSE: 15.2996
Probando: n_estimators=100, max_depth=5, min_samples_split=2
   Train RMSE: 16.6219, Val RMSE: 16.3719
Probando: n_estimators=100, max_depth=5, min_samples_split=5
   Train RMSE: 16.6219, Val RMSE: 16.3719
Probando: n_estimators=100, max_depth=10, min_samples_split=2
   Train RMSE: 13.2358, Val RMSE: 15.2685
Probando: n_estimators=100, max_depth=10, min_samples_split=5
   Train RMSE: 13.4108, Val RMSE: 15.2805


In [37]:

try:
    from xgboost import XGBRegressor
    xgb_available = True
except ImportError:
    print("XGBoost no disponible, usando GradientBoostingRegressor...")
    from sklearn.ensemble import GradientBoostingRegressor
    xgb_available = False

# Hiperparámetros optimizados para velocidad
n_estimators_list = [50, 100]
max_depth_list = [3, 6]
learning_rate_list = [0.1, 0.05]

print("Hiperparametros XGBoost:")
print(f"   n_estimators: {n_estimators_list}")
print(f"   max_depth: {max_depth_list}")
print(f"   learning_rate: {learning_rate_list}")

results_xgb = []

for n_estimators in n_estimators_list:
    for max_depth in max_depth_list:
        for learning_rate in learning_rate_list:
            print(f" Probando: n_estimators={n_estimators}, max_depth={max_depth}, learning_rate={learning_rate}")
            
            try:
                if xgb_available:
                    model = XGBRegressor(
                        n_estimators=n_estimators,
                        max_depth=max_depth,
                        learning_rate=learning_rate,
                        random_state=42,
                        n_jobs=-1,
                        verbosity=0
                    )
                else:
                    model = GradientBoostingRegressor(
                        n_estimators=n_estimators,
                        max_depth=max_depth,
                        learning_rate=learning_rate,
                        random_state=42
                    )
                
                model.fit(X_train, y_train.values.ravel())
                
                y_train_pred = model.predict(X_train)
                y_val_pred = model.predict(X_val)
                
                train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
                val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
                train_r2 = r2_score(y_train, y_train_pred)
                val_r2 = r2_score(y_val, y_val_pred)
                
                results_xgb.append({
                    'algorithm': 'XGBoost_Regressor' if xgb_available else 'GradientBoosting_Regressor',
                    'n_estimators': n_estimators,
                    'max_depth': max_depth,
                    'learning_rate': learning_rate,
                    'train_rmse': train_rmse,
                    'val_rmse': val_rmse,
                    'train_r2': train_r2,
                    'val_r2': val_r2
                })
                
                print(f"   Train RMSE: {train_rmse:.4f}, Val RMSE: {val_rmse:.4f}")
                
            except Exception as e:
                print(f"   Error: {e}")
                continue

results_svr = results_xgb

Hiperparametros XGBoost:
   n_estimators: [50, 100]
   max_depth: [3, 6]
   learning_rate: [0.1, 0.05]
 Probando: n_estimators=50, max_depth=3, learning_rate=0.1
   Train RMSE: 19.2564, Val RMSE: 18.9591
 Probando: n_estimators=50, max_depth=3, learning_rate=0.05
   Train RMSE: 20.7989, Val RMSE: 20.4633
 Probando: n_estimators=50, max_depth=6, learning_rate=0.1
   Train RMSE: 18.2184, Val RMSE: 18.2224
 Probando: n_estimators=50, max_depth=6, learning_rate=0.05
   Train RMSE: 18.5945, Val RMSE: 18.3925
 Probando: n_estimators=100, max_depth=3, learning_rate=0.1
   Train RMSE: 18.7204, Val RMSE: 18.3711
 Probando: n_estimators=100, max_depth=3, learning_rate=0.05
   Train RMSE: 19.2292, Val RMSE: 18.8922
 Probando: n_estimators=100, max_depth=6, learning_rate=0.1
   Train RMSE: 18.1059, Val RMSE: 18.2710
 Probando: n_estimators=100, max_depth=6, learning_rate=0.05
   Train RMSE: 18.2282, Val RMSE: 18.2178


In [33]:
n_estimators_list_clf = [50, 100]
max_depth_list_clf = [5, 10]
min_samples_split_list_clf = [2, 5]

print("Hiperparametros:")
print(f"   n_estimators: {n_estimators_list_clf}")
print(f"   max_depth: {max_depth_list_clf}")
print(f"   min_samples_split: {min_samples_split_list_clf}")

results_rf_clf = []

for n_estimators in n_estimators_list_clf:
    for max_depth in max_depth_list_clf:
        for min_samples_split in min_samples_split_list_clf:
            print(f"Probando: n_estimators={n_estimators}, max_depth={max_depth}, min_samples_split={min_samples_split}")
            
            try:
                model = RandomForestClassifier(
                    n_estimators=n_estimators,
                    max_depth=max_depth,
                    min_samples_split=min_samples_split,
                    random_state=42,
                    n_jobs=-1,
                    verbose=0
                )
                model.fit(X_train, y_train_class.values.ravel())
                
                y_train_pred = model.predict(X_train)
                y_val_pred = model.predict(X_val)
                
                train_accuracy = accuracy_score(y_train_class, y_train_pred)
                val_accuracy = accuracy_score(y_val_class, y_val_pred)
                
                results_rf_clf.append({
                    'algorithm': 'RandomForest_Classifier',
                    'n_estimators': n_estimators,
                    'max_depth': max_depth,
                    'min_samples_split': min_samples_split,
                    'train_accuracy': train_accuracy,
                    'val_accuracy': val_accuracy
                })
                
                print(f"  Train Accuracy: {train_accuracy:.4f}, Val Accuracy: {val_accuracy:.4f}")
                
            except Exception as e:
                print(f"   Error: {e}")
                continue

Hiperparametros:
   n_estimators: [50, 100]
   max_depth: [5, 10]
   min_samples_split: [2, 5]
Probando: n_estimators=50, max_depth=5, min_samples_split=2
  Train Accuracy: 0.7681, Val Accuracy: 0.7653
Probando: n_estimators=50, max_depth=5, min_samples_split=5
  Train Accuracy: 0.7685, Val Accuracy: 0.7659
Probando: n_estimators=50, max_depth=10, min_samples_split=2
  Train Accuracy: 0.8623, Val Accuracy: 0.8577
Probando: n_estimators=50, max_depth=10, min_samples_split=5
  Train Accuracy: 0.8619, Val Accuracy: 0.8572
Probando: n_estimators=100, max_depth=5, min_samples_split=2
  Train Accuracy: 0.7664, Val Accuracy: 0.7639
Probando: n_estimators=100, max_depth=5, min_samples_split=5
  Train Accuracy: 0.7663, Val Accuracy: 0.7638
Probando: n_estimators=100, max_depth=10, min_samples_split=2
  Train Accuracy: 0.8621, Val Accuracy: 0.8573
Probando: n_estimators=100, max_depth=10, min_samples_split=5
  Train Accuracy: 0.8615, Val Accuracy: 0.8567


In [44]:
# Combinar todos los resultados
all_results = results_rf + results_svr + results_rf_clf
results_df = pd.DataFrame(all_results)

best_results = []

for algorithm in ['RandomForest_Regressor', 'GradientBoosting_Regressor', 'RandomForest_Classifier']:
    algo_results = results_df[results_df['algorithm'] == algorithm]
    if not algo_results.empty:
        if algorithm == 'RandomForest_Classifier':
            best_idx = algo_results['val_accuracy'].idxmax()
            best_result = algo_results.loc[best_idx]
            print(f"\n🔹 {algorithm}:")
            print(f"   Mejor Validation Accuracy: {best_result['val_accuracy']:.4f}")
            print(f"   Hiperparametros: n_estimators={best_result['n_estimators']}, "
                  f"max_depth={best_result['max_depth']}, min_samples_split={best_result['min_samples_split']}")
            print(f"   Train Accuracy: {best_result['train_accuracy']:.4f}")
            
        else:
            best_idx = algo_results['val_rmse'].idxmin()
            best_result = algo_results.loc[best_idx]
            print(f"\n🔹 {algorithm}:")
            print(f"   • Mejor Validation RMSE: {best_result['val_rmse']:.4f}")
            if 'val_r2' in best_result:
                print(f"   • Mejor Validation R²: {best_result['val_r2']:.4f}")
            
            if algorithm == 'RandomForest_Regressor':
                print(f"   Hiperparametros: n_estimators={best_result['n_estimators']}, "
                      f"max_depth={best_result['max_depth']}, min_samples_split={best_result['min_samples_split']}")
                print(f"   Train RMSE: {best_result['train_rmse']:.4f}")
                print(f"   Overfitting (diferencia): {best_result['train_rmse'] - best_result['val_rmse']:.4f}")
            
            else:  # GradientBoosting_Regressor
                print(f"   Hiperparametros: n_estimators={best_result['n_estimators']}, "
                      f"max_depth={best_result['max_depth']}, learning_rate={best_result['learning_rate']}")
                print(f"   Train RMSE: {best_result['train_rmse']:.4f}")
                if 'train_r2' in best_result:
                    print(f"   Train R2: {best_result['train_r2']:.4f}")
        
        best_results.append(best_result)

# Crear tabla resumen comparativa
comparison_df = pd.DataFrame(best_results)

# Seleccionar y ordenar columnas para mejor presentación
columns_to_show = ['algorithm', 'val_rmse', 'val_r2', 'val_accuracy']
available_columns = [col for col in columns_to_show if col in comparison_df.columns]

display(comparison_df[available_columns].round(4))

# Analisis comparativo adicional

if len(best_results) >= 2:
    # Encontrar mejor algoritmo de regresión
    regressors = [r for r in best_results if r['algorithm'] != 'RandomForest_Classifier']
    if regressors:
        best_regressor = min(regressors, key=lambda x: x['val_rmse'])
        print(f" mejor algoritmo de regresion: {best_regressor['algorithm']}")
        print(f"   Validacion RMSE: {best_regressor['val_rmse']:.4f}")
        if 'val_r2' in best_regressor:
            print(f"   Validacion R2: {best_regressor['val_r2']:.4f}")
    
    # Información del clasificador
    classifier = [r for r in best_results if r['algorithm'] == 'RandomForest_Classifier']
    if classifier:
        print(f"Algoritmo de clasificacion: {classifier[0]['algorithm']}")
        print(f"    validacion Accuracy: {classifier[0]['val_accuracy']:.4f}")



🔹 RandomForest_Regressor:
   • Mejor Validation RMSE: 15.2685
   • Mejor Validation R²: 0.8487
   Hiperparametros: n_estimators=100, max_depth=10, min_samples_split=2.0
   Train RMSE: 13.2358
   Overfitting (diferencia): -2.0327

🔹 RandomForest_Classifier:
   Mejor Validation Accuracy: 0.8577
   Hiperparametros: n_estimators=50, max_depth=10, min_samples_split=2.0
   Train Accuracy: 0.8623


,algorithm,val_rmse,val_r2,val_accuracy
6,RandomForest_Regressor,15.2685,0.8487,NaN
18,RandomForest_Classifier,NaN,NaN,0.8577


 mejor algoritmo de regresion: RandomForest_Regressor
   Validacion RMSE: 15.2685
   Validacion R2: 0.8487
Algoritmo de clasificacion: RandomForest_Classifier
    validacion Accuracy: 0.8577


En la comparativa entre Gradient Boosting y Random Forest para regresion, Gradient Boosting demostro un rendimiento ligeramente superior con un RMSE de validacion de 15.18 frente a 15.27 del Random Forest, ademas de mostrar un mejor control del overfitting. La diferencia de 0.75 unidades entre el error de entrenamiento y validacion en Gradient Boosting versus 2.03 en Random Forest indica que el primero generaliza mejor con nuevos datos. La configuracion optima de Gradient Boosting con learning rate de 0.05 y profundidad maxima de 6 logro el balance ideal entre capacidad predictiva y estabilidad.

En cuanto al clasificador de rentabilidad, el Random Forest alcanzo una precision destacada del 85.77% en validacion, manteniendo consistencia con los datos de entrenamiento. Esta alta efectividad en clasificar peliculas como de alta o baja rentabilidad lo convierte en una herramienta confiable para decisiones binarias de inversion. Los parametros optimos con profundidad maxima de 10 permitieron capturar patrones complejos sin comprometer la generalizacion, demostrando que la clasificacion basada en caracteristicas pre-estreno puede predecir con alta certeza el exito comercial.

Para estimaciones puntuales de ROI se recomienda Gradient Boosting por su mayor precision numerica, mientras que para clasificacion de riesgo el Random Forest ofrece decisiones categoricas con alta confiabilidad. Ambos algoritmos demostraron ser efectivos, con Gradient Boosting mostrando ventaja en regresion gracias a su enfoque de correccion secuencial de errores, y Random Forest manteniendo su robustez tanto en regresion como en clasificacion


In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

# Combinar entrenamiento + validación para aprovechar todos los datos
X_full = pd.concat([X_train, X_val])
y_full = pd.concat([y_train, y_val])

# Entrenar el mejor modelo encontrado
best_model = GradientBoostingRegressor(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.05,
    random_state=42
)

best_model.fit(X_full, y_full.values.ravel())

print("\n Mejor modelo (GradientBoostingRegressor) entrenado correctamente.")


ejemplos_nuevos = pd.DataFrame({
    'BudgetUSD': [0.6, -0.2, 0.4, 1.2, 2.0],
    'One_Week_SalesUSD': [0.5, 0.0, 0.8, 2.5, 4.2]
})

print("\n--- Nuevos 5 registros para predicción ---")
print(ejemplos_nuevos)

# Realizar predicciones
predicciones = best_model.predict(ejemplos_nuevos)

print("\n--- Predicciones del modelo (ROI estimado) ---")
for i, pred in enumerate(predicciones, start=1):
    print(f"Registro {i}: ROI predicho = {pred:.4f}")



✅ Mejor modelo (GradientBoostingRegressor) entrenado correctamente.

--- Nuevos 5 registros para predicción ---
   BudgetUSD  One_Week_SalesUSD
0        0.6                0.5
1       -0.2                0.0
2        0.4                0.8
3        1.2                2.5
4        2.0                4.2

--- Predicciones del modelo (ROI estimado) ---
Registro 1: ROI predicho = 1.8961
Registro 2: ROI predicho = 2.2376
Registro 3: ROI predicho = 2.5262
Registro 4: ROI predicho = 1.0414
Registro 5: ROI predicho = -0.3462


In [5]:
print("Columnas esperadas:", list(X_full.columns))
print("Columnas usadas en las nuevas películas:", list(nuevas_peliculas_num.columns))
print("Dimensiones de entrada:", nuevas_peliculas_num.shape)


Columnas esperadas: ['BudgetUSD', 'One_Week_SalesUSD']
Columnas usadas en las nuevas películas: ['BudgetUSD', 'One_Week_SalesUSD']
Dimensiones de entrada: (5, 2)
